In [1]:
# import re

# def extract_questions(text):
#     lines = text.split("\n")
#     questions = []
#     for line in lines:
#         line = line.strip()
#         if re.match(r"^(\d+\.|-|\*)\s+", line):
#             questions.append(re.sub(r"^(\d+\.|-|\*)\s+", "", line))
#         elif line.endswith("?"):
#             questions.append(line)
#     return questions
  

# import ollama

# client = ollama.Client()
# MODEL_NAME = "llama3.2:1b"


# file_content = """// db.js
# import mysql from "mysql2/promise";

# export const pool = mysql.createPool({
#   host: "localhost",
#   user: "root",
#   password: "YOUR_PASSWORD",
#   database: "hello",
#   connectionLimit: 10,
# });
# """

# prompt = (
#     "Read the following text and generate a list of clear, helpful questions "
#     "that someone might ask to better understand it.\n\n"
#     f"--- TEXT START ---\n{file_content}\n--- TEXT END ---\n\n"
#     "Questions:"
# )

# response = client.generate(model=MODEL_NAME, prompt=prompt)
# raw_output = response["response"]
# print(raw_output)
# questions = extract_questions(raw_output)
# questions

## Section: Creation of input.json
### Run below only once because it creates teh input.json

In [3]:
# import os
# import json
# import ollama

# client = ollama.Client()
# MODEL_NAME = "llama3.2:1b"

# # Directory containing your code files
# directory_path = "./BI-backend"  # <-- change to your folder
# output_file = "input.json"

# input_data = []

# for filename in os.listdir(directory_path):
#     file_path = os.path.join(directory_path, filename)
    
#     if os.path.isfile(file_path):
#         with open(file_path, "r", encoding="utf-8") as f:
#             file_content = f.read()
        
#         prompt = (
#             "Read the following text and generate a list of clear, concise, helpful questions "
#             "that someone might ask to better understand it.\n\n"
#             f"--- TEXT START ({filename}) ---\n{file_content}\n--- TEXT END ---\n\n"
#             "Return only a JSON array of questions (strings)."
#         )
        
#         response = client.generate(model=MODEL_NAME, prompt=prompt)
#         raw_output = response["response"].strip()
        
#         # Try to parse JSON output safely
#         try:
#             questions = json.loads(raw_output)
#             if not isinstance(questions, list):
#                 raise ValueError
#         except Exception:
#             # fallback: split by newlines if JSON fails
#             questions = [q.strip("- ").strip() for q in raw_output.split("\n") if q.strip()]
        
#         # Append to input data
#         input_data.append({
#             "file_name": filename,
#             "code": file_content,
#             "questions": questions
#         })
        
#         print(f"Generated {len(questions)} questions for {filename}")

# # Save all data to input.json
# with open(output_file, "w", encoding="utf-8") as f:
#     json.dump(input_data, f, indent=2)

# print(f"\nAll files processed. Input JSON saved to {output_file}")


In [4]:
# Now the input.json is created

OpenAI evaluation

In [5]:
# import json
# import re
# import hashlib
# from typing import Dict, List
# from openai import OpenAI

# # =========================
# # CONFIG
# # =========================
# MODEL = "gpt-4.1-mini"
# LLM_WEIGHT = 0.6
# RULE_WEIGHT = 0.4
# MIN_LENGTH = 5

# client = OpenAI()

# # =========================
# # RULE-BASED SCORING
# # =========================
# def rule_score(code: str, question: str) -> Dict[str, int]:
#     score = {"relevance": 1, "specificity": 1, "depth": 1, "clarity": 1}

#     identifiers = set(re.findall(r"\b[a-zA-Z_][a-zA-Z0-9_]*\b", code))
#     literals = re.findall(r"['\"](.*?)['\"]|\b\d+\b", code)

#     if any(id.lower() in question.lower() for id in identifiers):
#         score["relevance"] += 2
#         score["specificity"] += 1

#     if any(str(lit) in question for lit in literals):
#         score["specificity"] += 2

#     if re.search(r"\b(why|difference|trade[- ]off|advantage|impact)\b", question, re.I):
#         score["depth"] += 3

#     if len(question.split()) >= MIN_LENGTH and question.endswith("?"):
#         score["clarity"] += 3

#     return {k: min(v, 5) for k, v in score.items()}

# # =========================
# # LLM-BASED SCORING
# # =========================
# def llm_score(code: str, question: str) -> Dict[str, int]:
#     prompt = f"""
# You are evaluating the quality of a question about the following code.

# CODE:
# {code}

# QUESTION:
# {question}

# Score from 1–5 on:
# - relevance
# - specificity
# - depth
# - clarity

# Return JSON only.
# """

#     response = client.responses.create(
#         model=MODEL,
#         input=prompt,
#         temperature=0
#     )

#     text = response.output_text
#     return json.loads(text)

# # =========================
# # COMBINED SCORING
# # =========================
# def combine_scores(rule: Dict, llm: Dict) -> Dict:
#     combined = {}
#     total = 0

#     for key in ["relevance", "specificity", "depth", "clarity"]:
#         combined[key] = round(
#             RULE_WEIGHT * rule[key] + LLM_WEIGHT * llm[key], 2
#         )
#         total += combined[key]

#     combined["total"] = round(total, 2)
#     return combined

# # =========================
# # MAIN PIPELINE
# # =========================
# def score_file(data: Dict) -> Dict:
#     results = []

#     for q in data["questions"]:
#         rule = rule_score(data["code"], q)
#         llm = llm_score(data["code"], q)
#         final = combine_scores(rule, llm)

#         results.append({
#             "question": q,
#             "rule_score": rule,
#             "llm_score": llm,
#             "final_score": final
#         })

#     avg = round(sum(r["final_score"]["total"] for r in results) / len(results), 2)

#     return {
#         "file": data["file_name"],
#         "average_score": avg,
#         "questions": results
#     }

# # =========================
# # RUN
# # =========================
# if __name__ == "__main__":
#     with open("input.json") as f:
#         files = json.load(f)

#     reports = [score_file(file) for file in files]

#     with open("report.json", "w") as f:
#         json.dump(reports, f, indent=2)

#     print("Scoring complete → report.json")


Gemini version

In [6]:
import json
import re
from typing import Dict, List
import google.generativeai as genai

from dotenv import load_dotenv
import os
import google.generativeai as genai




load_dotenv()  # loads variables from .env
api_key = os.getenv("GENERATIVEAI_API_KEY")
genai.configure(api_key=api_key)


c:\Users\HP\Documents\GitHub\25-26J-087\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\HP\AppData\Local\Temp\ipykernel_22700\3505649446.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


# Section: Available models

In [ ]:

for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-exp-1206
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image-preview
models/gemini-2.5-flash-image
models/gemini-2.5-flash-preview-09-2025
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-prev

In [8]:
MODEL = "models/gemini-2.5-flash"  # Free-tier Gemini model
LLM_WEIGHT = 0.6
RULE_WEIGHT = 0.4
MIN_LENGTH = 5
# =========================
# LLM-BASED SCORING (Gemini)
# =========================
def llm_score(code: str, question: str) -> Dict[str, int]:
    prompt = f"""
You are evaluating the quality of a question about the following code.

CODE:
{code}

QUESTION:
{question}

Score from 1–5 on:
- relevance
- specificity
- depth
- clarity

Return JSON only.
"""

    # Create the model object
    model = genai.GenerativeModel(MODEL)  

    # Send a single content generation request
    response = model.generate_content(prompt)

    # Get the text string
    text_output = response.text

    try:
        return json.loads(text_output)
    except json.JSONDecodeError:
        return {
            "relevance": 3,
            "specificity": 3,
            "depth": 3,
            "clarity": 3
        }

simple usage of above

In [9]:
# # Example Python code snippet
# code_snippet = """
# def add(a, b):
#     return a + b
# """

# # Example question about the code
# question_text = "How can I modify this function to handle adding multiple numbers?"

# # Call the LLM scoring function
# scores = llm_score(code_snippet, question_text)

# print(scores)


In [ ]:
# =========================
# RULE-BASED SCORING
# =========================
def rule_score(code: str, question: str) -> Dict[str, int]:
    score = {"relevance": 1, "specificity": 1, "depth": 1, "clarity": 1}

    identifiers = set(re.findall(r"\b[a-zA-Z_][a-zA-Z0-9_]*\b", code))
    literals = re.findall(r"['\"](.*?)['\"]|\b\d+\b", code)

    if any(id.lower() in question.lower() for id in identifiers):
        score["relevance"] += 2
        score["specificity"] += 1

    if any(str(lit) in question for lit in literals):
        score["specificity"] += 2

    if re.search(r"\b(why|difference|trade[- ]off|advantage|impact)\b", question, re.I):
        score["depth"] += 3

    if len(question.split()) >= MIN_LENGTH and question.endswith("?"):
        score["clarity"] += 3

    return {k: min(v, 5) for k, v in score.items()}


def combine_scores(rule: Dict, llm: Dict) -> Dict:
    combined = {}
    total = 0

    for key in ["relevance", "specificity", "depth", "clarity"]:
        combined[key] = round(
            RULE_WEIGHT * rule[key] + LLM_WEIGHT * llm[key], 2
        )
        total += combined[key]

    combined["total"] = round(total, 2)
    return combined

def score_file(data: Dict) -> Dict:
    results = []

    for q in data["questions"]:
        rule = rule_score(data["code"], q)
        llm = llm_score(data["code"], q)
        final = combine_scores(rule, llm)

        results.append({
            "question": q,
            "rule_score": rule,
            "llm_score": llm,
            "final_score": final
        })

    avg = round(sum(r["final_score"]["total"] for r in results) / len(results), 2)

    return {
        "file": data["file_name"],
        "average_score": avg,
        "questions": results
    }


In [ ]:
if __name__ == "__main__":
    with open("input.json") as f:
        files = json.load(f)

    reports = [score_file(file) for file in files]

    with open("report.json", "w") as f:
        json.dump(reports, f, indent=2)

    print("Scoring complete → report.json")


Scoring complete → report.json
